# ⚛️ Notebook 4: Quantum Machine Learning

Demonstrates Quantum Kernel SVM and Variational Quantum Classifier (VQC)
on a small subset of micro-Doppler features.

> **Note**: Quantum simulation is exponentially costly. We use ≤ 4 qubits
> and ≤ 300 samples for practical simulation on CPU.

## Setup

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler

try:
    import pennylane as qml
    print(f"PennyLane version: {qml.__version__}")
except ImportError:
    print("PennyLane not installed. Run: pip install pennylane pennylane-lightning")
    raise

from Dataset.loader import load_dataset, get_iq_matrix, get_labels, get_train_test_split
from Preprocessing.features import extract_features
from Preprocessing.normalize import fit_feature_scaler
from Quantum_ML.qkernel_svm import QuantumKernelSVM
from Quantum_ML.vqc import VariationalQuantumClassifier
from Evaluation.metrics import compute_metrics, print_metrics

plt.style.use('dark_background')
COLORS = ['#58a6ff', '#3fb950', '#f78166']
print("✅ Ready")

## 1. Prepare Small Quantum-Friendly Dataset

Reduce dimensionality to `n_qubits=4` via PCA, then scale to `[0, π]`.

In [ ]:
N_QUBITS = 4
N_TRAIN  = 150   # keep small for simulation speed
N_TEST   = 50

print(f"Loading dataset ({N_TRAIN + N_TEST} samples)...")
df = load_dataset('../helicopter_microdoppler_dataset.csv', nrows=N_TRAIN + N_TEST + 200)
X_iq_train, X_iq_test, y_train, y_test = get_train_test_split(df, test_size=N_TEST/(N_TRAIN+N_TEST))

# Feature extraction
X_train_feat = extract_features(X_iq_train[:N_TRAIN])
X_test_feat  = extract_features(X_iq_test[:N_TEST])

# Standard scaling
scaler = fit_feature_scaler(X_train_feat)
X_tr_sc = scaler.transform(X_train_feat)
X_te_sc = scaler.transform(X_test_feat)

# PCA → 4 dimensions
pca = PCA(n_components=N_QUBITS, random_state=42)
X_tr_pca = pca.fit_transform(X_tr_sc)
X_te_pca = pca.transform(X_te_sc)

print(f"PCA explained variance: {pca.explained_variance_ratio_.sum():.3f} ({N_QUBITS} components)")

# Scale to [0, π] for angle embedding
angle_scaler = MinMaxScaler(feature_range=(0, np.pi))
X_tr_q = angle_scaler.fit_transform(X_tr_pca)
X_te_q = angle_scaler.transform(X_te_pca)

print(f"Quantum feature shape: {X_tr_q.shape} (train), {X_te_q.shape} (test)")
print(f"Value range: [{X_tr_q.min():.3f}, {X_tr_q.max():.3f}]")

## 2. Quantum Kernel SVM

In [ ]:
print("Training Quantum Kernel SVM...")
qksvm = QuantumKernelSVM(n_qubits=N_QUBITS, C=1.0)
qksvm.fit(X_tr_q, y_train[:N_TRAIN])
acc_qksvm = qksvm.score(X_te_q, y_test[:N_TEST])
y_pred_qksvm = qksvm.predict(X_te_q)
m_qksvm = compute_metrics(y_test[:N_TEST], y_pred_qksvm)
print_metrics(m_qksvm, 'Quantum Kernel SVM')

## 3. Variational Quantum Classifier (VQC)

In [ ]:
print("Training VQC (this may take 2-5 minutes)...")
vqc = VariationalQuantumClassifier(
    n_qubits=N_QUBITS, n_layers=2, n_classes=3,
    lr=0.02, n_epochs=20, batch_size=16
)
vqc.fit(X_tr_q, y_train[:N_TRAIN])
y_pred_vqc = vqc.predict(X_te_q)
m_vqc = compute_metrics(y_test[:N_TEST], y_pred_vqc)
print_metrics(m_vqc, 'VQC')

## 4. Classical vs Quantum Comparison

In [ ]:
from Classical_ML.train import train_svm
from sklearn.preprocessing import StandardScaler

# Classical SVM on same 4D PCA features for fair comparison
classical_svm = train_svm(X_tr_q, y_train[:N_TRAIN])
y_pred_csvm = classical_svm.predict(X_te_q)
m_csvm = compute_metrics(y_test[:N_TEST], y_pred_csvm)

results = {
    'Classical SVM (4-D PCA)': m_csvm,
    'Quantum Kernel SVM':       m_qksvm,
    'VQC (2 layers)':           m_vqc,
}

print(f"\n{'Model':<30} {'Accuracy':>10} {'Macro-F1':>10}")
print("─" * 55)
for name, m in results.items():
    print(f"{name:<30} {m['accuracy']:>10.4f} {m['macro_f1']:>10.4f}")

# Bar chart comparison
fig, ax = plt.subplots(figsize=(10, 6))
names = list(results.keys())
accs  = [results[n]['accuracy'] for n in names]
f1s   = [results[n]['macro_f1'] for n in names]

x = np.arange(len(names))
w = 0.35
bars1 = ax.bar(x - w/2, accs, w, label='Accuracy', color=COLORS[0], alpha=0.85)
bars2 = ax.bar(x + w/2, f1s,  w, label='Macro-F1', color=COLORS[1], alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(names, fontsize=11)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Classical vs Quantum ML — Performance on 4-D PCA Features', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.2, axis='y')
ax.axhline(y=1/3, color='white', linestyle='--', alpha=0.4, label='Random baseline')

for bar in list(bars1) + list(bars2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('quantum_vs_classical.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print("✅ Quantum ML notebook complete!")